# ADHD-200 fMRI motion scrubbing and atlas-scale modules

Third-stage sensitivity analysis for the locked A424 cohort. It maps native-TR framewise displacement to the saved ~1 Hz parcel series, conservatively censors interpolation intervals touching high-motion frames, and repeats strict nested leave-one-site-out (LOSO) evaluation. It also tests low-dimensional atlas-coordinate modules. Research use only; this is not a clinical diagnostic system.

In [ ]:
%pip install -q scikit-learn scipy requests

In [ ]:
from google.colab import drive
drive.mount('<DRIVE_MOUNT>')
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import io, re, requests
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.cluster import KMeans
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import roc_auc_score

ROOT=Path('<DATA_DIR>')
BRAINLM=ROOT/'fmri/brainlm_a424'
BENCH=ROOT/'fmri/strict_loso_benchmark'
OUT=BENCH/'stage3_scrubbing_network'
OUT.mkdir(parents=True,exist_ok=True)
TS_DIR=BRAINLM/'timeseries_raw'
INDEX=BRAINLM/'fmriprep_aroma_index.csv'
REPO=Path('/content/BrainLM')
if not REPO.exists():
    !git clone -q https://github.com/vandijklab/BrainLM.git /content/BrainLM

primary=pd.read_csv(BENCH/'locked_primary_cohort_409_with_motion.csv',dtype={'subject_id':str})
idx=pd.read_csv(INDEX,dtype={'subject_id':str})
qc=pd.read_csv(BRAINLM/'a424_extraction_qc.csv',dtype={'subject_id':str})
bank=np.load(BENCH/'a424_feature_bank.npz',allow_pickle=True)
assert np.array_equal(bank['subject_id'].astype(str),primary.subject_id.to_numpy())
X_fcsummary=bank['fc_summary'].astype(np.float32)
X_edges=bank['edges'].astype(np.float32)
X_conf=primary[['age','sex_male','mean_fd','max_fd','pct_fd_gt_0p2','mean_dvars','n_volumes','qc_rest']].to_numpy(float)
BASE_URL='https://fcp-indi.s3.amazonaws.com/'

def confounds_key(bold_key):
    folder,name=bold_key.rsplit('/',1)
    stem=re.sub(r'_space-[^_]+_desc-smoothAROMAnonaggr_bold\.nii\.gz$','',name)
    return f'{folder}/{stem}_desc-confounds_regressors.tsv'

print('Locked cohort:',len(primary),'subjects across',primary.site.nunique(),'sites')

## 1. Conservative motion censoring

A native frame is marked bad when FD > 0.2 mm. One preceding and two following native frames are also marked. A 1 Hz sample is removed when either native frame contributing to its interpolation interval is bad. Subjects retaining fewer than 100 one-second samples are excluded from this sensitivity cohort. This is conservative post-resampling censoring, not a substitute for rebuilding parcel time series directly from censored native volumes.

In [ ]:
site_tr=qc.groupby('site').tr.median().to_dict()
tr_by_sid=qc.dropna(subset=['tr']).drop_duplicates('subject_id',keep='last').set_index('subject_id').tr.to_dict()
key_by_sid=idx[idx.status.eq('ok')].drop_duplicates('subject_id').set_index('subject_id').s3_key.to_dict()

def fetch_fd(sid):
    key=confounds_key(key_by_sid[sid])
    r=requests.get(BASE_URL+key,timeout=90); r.raise_for_status()
    c=pd.read_csv(io.StringIO(r.text),sep='\t')
    return sid,c.framewise_displacement.fillna(0).to_numpy(float)

fd_map={}; download_errors={}
with ThreadPoolExecutor(max_workers=12) as ex:
    jobs={ex.submit(fetch_fd,sid):sid for sid in primary.subject_id}
    for j,f in enumerate(as_completed(jobs),1):
        sid=jobs[f]
        try: fd_map[sid]=f.result()[1]
        except Exception as e: download_errors[sid]=repr(e)
        if j%50==0: print('confounds',j,'/',len(jobs))

fc_list=[]; feat_list=[]; rows=[]
for _,row in primary.iterrows():
    sid=row.subject_id; site=row.site
    try:
        ts=np.asarray(np.load(TS_DIR/f'{sid}.npy'),dtype=np.float32)
        fd=fd_map[sid]; tr=float(tr_by_sid.get(sid,site_tr[site]))
        bad=fd>0.2; expanded=bad.copy()
        expanded[:-1] |= bad[1:]
        expanded[1:] |= bad[:-1]
        expanded[2:] |= bad[:-2]
        t=np.arange(ts.shape[0],dtype=float)
        left=np.clip(np.floor(t/tr).astype(int),0,len(fd)-1)
        right=np.clip(np.ceil(t/tr).astype(int),0,len(fd)-1)
        keep=~(expanded[left]|expanded[right])
        if keep.sum()<100: raise ValueError(f'only {keep.sum()} retained 1Hz samples')
        c=np.corrcoef(ts[keep],rowvar=False).astype(np.float32)
        c=np.nan_to_num(c,nan=0.0,posinf=0.0,neginf=0.0); np.fill_diagonal(c,0)
        z=np.arctanh(np.clip(c,-0.999999,0.999999)); np.fill_diagonal(z,0)
        fc_list.append(z)
        feat_list.append(np.r_[z.mean(1),z.std(1)].astype(np.float32))
        rows.append({'subject_id':sid,'site':site,'label':int(row.label),'native_tr':tr,
                     'native_n':len(fd),'onehz_n':len(ts),'retained_1hz':int(keep.sum()),
                     'retained_fraction':float(keep.mean()),'status':'ok'})
    except Exception as e:
        rows.append({'subject_id':sid,'site':site,'label':int(row.label),'status':repr(e)})

scrub_qc=pd.DataFrame(rows)
scrub_qc.to_csv(OUT/'motion_scrubbing_qc.csv',index=False)
ok_ids=[r['subject_id'] for r in rows if r['status']=='ok']
pos={sid:i for i,sid in enumerate(primary.subject_id)}
sel=np.array([pos[sid] for sid in ok_ids])
clean=primary.iloc[sel].reset_index(drop=True)
FC_scrub=np.stack(fc_list); X_scrub=np.stack(feat_list)
X_conf_clean=X_conf[sel]; X_fc_clean=X_fcsummary[sel]
np.save(OUT/'scrubbed_fc_roi_summary.npy',X_scrub)
clean.to_csv(OUT/'clean_cohort_subjects.csv',index=False)
display(pd.crosstab(scrub_qc.site,scrub_qc.status.eq('ok')))
display(scrub_qc.loc[scrub_qc.status.eq('ok'),'retained_fraction'].describe())
print('Clean sensitivity cohort:',len(clean))

## 2. Strict nested LOSO comparison

Every imputer, univariate selector, scaler, regularization choice, and classifier is fitted inside the appropriate training fold. The same retained subjects are used for all comparisons below.

In [ ]:
def transform_fold(Xtr,ytr,Xte,k=None):
    imp=SimpleImputer(strategy='median').fit(Xtr)
    a=imp.transform(Xtr); b=imp.transform(Xte)
    if k is not None and a.shape[1]>k:
        selector=SelectKBest(f_classif,k=k).fit(a,ytr)
        a=selector.transform(a); b=selector.transform(b)
    sc=StandardScaler().fit(a)
    return sc.transform(a),sc.transform(b)

def nested_loso(X,name,k=None,Cs=(0.01,0.1,1.0)):
    y=clean.label.to_numpy(); g=clean.site.astype(str).to_numpy()
    logo=LeaveOneGroupOut(); oof=np.full(len(y),np.nan); rows=[]
    for tr,te in logo.split(X,y,g):
        scores={C:[] for C in Cs}
        for itr,iva in logo.split(X[tr],y[tr],g[tr]):
            a,b=transform_fold(X[tr][itr],y[tr][itr],X[tr][iva],k)
            for C in Cs:
                clf=LogisticRegression(C=C,class_weight='balanced',max_iter=3000).fit(a,y[tr][itr])
                scores[C].append(roc_auc_score(y[tr][iva],clf.predict_proba(b)[:,1]))
        best=max(Cs,key=lambda C:np.nanmean(scores[C]))
        a,b=transform_fold(X[tr],y[tr],X[te],k)
        clf=LogisticRegression(C=best,class_weight='balanced',max_iter=3000).fit(a,y[tr])
        p=clf.predict_proba(b)[:,1]; oof[te]=p
        rows.append({'model':name,'site':g[te][0],'n':len(te),'auc':roc_auc_score(y[te],p),'C':best})
    by=pd.DataFrame(rows)
    summary={'model':name,'n':len(y),'macro_auc':by.auc.mean(),
             'weighted_macro_auc':np.average(by.auc,weights=by.n),'pooled_oof_auc':roc_auc_score(y,oof)}
    pred=pd.DataFrame({'subject_id':clean.subject_id,'site':g,'y':y,'prob':oof,'model':name})
    return summary,by,pred

summaries=[]; by_site=[]; predictions=[]
for X,name,k in [(X_conf_clean,'clean cohort confounds',None),
                 (X_fc_clean,'clean cohort original FC summary',200),
                 (X_scrub,'clean cohort scrubbed FC summary',200),
                 (np.c_[X_conf_clean,X_scrub],'clean cohort confounds + scrubbed FC',200)]:
    s,b,p=nested_loso(X,name,k); summaries.append(s); by_site.append(b); predictions.append(p)
basic_summary=pd.DataFrame(summaries).sort_values('macro_auc',ascending=False)
basic_summary.to_csv(OUT/'stage3_basic_summary.csv',index=False)
pd.concat(by_site).to_csv(OUT/'stage3_basic_by_site.csv',index=False)
pd.concat(predictions).to_csv(OUT/'stage3_basic_predictions.csv',index=False)
display(basic_summary)

## 3. Atlas-coordinate module sensitivity

AAL-424 does not ship with canonical functional-network labels in the BrainLM repository. To avoid inventing labels, 12 balanced spatial modules are learned once from the atlas coordinates only, independent of subjects and outcomes. Mean within- and between-module Fisher-z connectivity yields 78 features.

In [ ]:
coords=np.loadtxt(REPO/'toolkit/atlases/A424_Coordinates.dat')[:,1:4]
labels=KMeans(n_clusters=12,random_state=0,n_init=50).fit_predict(StandardScaler().fit_transform(coords))
tri=np.triu_indices(424,1)
FC_orig=np.zeros((len(clean),424,424),dtype=np.float32)
FC_orig[:,tri[0],tri[1]]=X_edges[sel]
FC_orig[:,tri[1],tri[0]]=X_edges[sel]

def block_features(FC,labels):
    k=int(labels.max()+1)
    a=np.minimum(labels[tri[0]],labels[tri[1]]); b=np.maximum(labels[tri[0]],labels[tri[1]])
    raw=a*k+b; uniq=np.unique(raw); lookup={u:i for i,u in enumerate(uniq)}
    code=np.array([lookup[u] for u in raw]); count=np.bincount(code,minlength=len(uniq)).astype(float)
    mapping=sparse.csr_matrix((1.0/count[code],(np.arange(len(code)),code)),shape=(len(code),len(uniq)))
    return np.asarray(FC[:,tri[0],tri[1]]@mapping,dtype=np.float32)

X_mod_orig=block_features(FC_orig,labels); X_mod_scrub=block_features(FC_scrub,labels)
pd.DataFrame({'parcel':np.arange(1,425),'x':coords[:,0],'y':coords[:,1],'z':coords[:,2],
              'spatial_module':labels}).to_csv(OUT/'coordinate_module_assignments.csv',index=False)

module_s=[]; module_b=[]; module_p=[]
for X,name in [(X_mod_orig,'clean cohort original spatial-module FC'),
               (X_mod_scrub,'clean cohort scrubbed spatial-module FC'),
               (np.c_[X_conf_clean,X_mod_scrub],'clean cohort confounds + scrubbed spatial-module FC')]:
    s,b,p=nested_loso(X,name); module_s.append(s); module_b.append(b); module_p.append(p)
module_summary=pd.DataFrame(module_s).sort_values('macro_auc',ascending=False)
module_summary.to_csv(OUT/'stage3_spatial_module_summary.csv',index=False)
pd.concat(module_b).to_csv(OUT/'stage3_spatial_module_by_site.csv',index=False)
pd.concat(module_p).to_csv(OUT/'stage3_spatial_module_predictions.csv',index=False)
display(module_summary)

## 4. Paired site-stratified bootstrap

The bootstrap resamples subjects within every site and diagnosis class, preserving the evaluation structure and allowing paired comparisons between representations.

In [ ]:
pred=pd.concat(predictions)
wide=pred.pivot_table(index=['subject_id','site','y'],columns='model',values='prob').reset_index()
models=['clean cohort confounds','clean cohort original FC summary','clean cohort scrubbed FC summary']
rng=np.random.default_rng(20260904); boots=[]
for _ in range(2000):
    values={m:[] for m in models}
    for site,d in wide.groupby('site'):
        take=[]
        for _,dd in d.groupby('y'):
            take.extend(rng.choice(dd.index.to_numpy(),size=len(dd),replace=True))
        bb=wide.loc[take]
        for m in models: values[m].append(roc_auc_score(bb.y,bb[m]))
    boots.append({m:np.mean(values[m]) for m in models})
boot=pd.DataFrame(boots); ci=[]
for m in models:
    ci.append({'contrast':m,'estimate':boot[m].mean(),'ci_low':boot[m].quantile(.025),'ci_high':boot[m].quantile(.975)})
for a,b in [(models[2],models[1]),(models[2],models[0])]:
    delta=boot[a]-boot[b]
    ci.append({'contrast':f'{a} minus {b}','estimate':delta.mean(),
               'ci_low':delta.quantile(.025),'ci_high':delta.quantile(.975)})
ci=pd.DataFrame(ci); ci.to_csv(OUT/'stage3_paired_bootstrap_ci.csv',index=False)
display(ci)

## Observed result and decision

The conservative censoring rule retained 246 of 409 subjects across all seven evaluable mixed-class sites. On this identical cohort, strict nested LOSO macro AUC was **0.622** for age/sex/motion/QC confounds, **0.532** for the original FC summary, and **0.506** for the scrubbed FC summary. The paired bootstrap difference for scrubbed minus original FC was **-0.026** (95% CI **-0.149 to 0.072**). Spatial-module FC obtained 0.493 before censoring and 0.441 after censoring; adding it to confounds obtained 0.541.

Conclusion: aggressive post-resampling censoring does not recover a stable cross-site ADHD signal and discards about 40% of subjects. The next high-value step is not a larger Transformer. It is to rebuild FC from native-time series with nuisance regression and censoring before resampling, then evaluate a prespecified ADHD-oriented feature set under the same LOSO protocol. If native parcel series cannot be regenerated, this branch should stop here and the project should report the negative imaging result transparently.